In [15]:
import pandas as pd
import numpy as np
import sqlite3
import pyarrow as pa
import pyarrow.parquet as pq
from skrub import TableReport
from skrub import Cleaner
from pathlib import Path
from typing import Dict, List, Optional, Tuple
import pprint
from tqdm import tqdm
import os
import shutil

## Create database

In [16]:
version = "0.0.3"

In [17]:
chunk1 = pd.read_csv("Data/human/salmon.merged.gene_counts1.tsv", index_col = ['gene_id', 'gene_name'], sep = "\t")
chunk2 = pd.read_csv("Data/human/salmon.merged.gene_counts2.tsv", index_col = ['gene_id', 'gene_name'], sep = "\t")

In [18]:
merged_chunks = pd.concat([chunk1, chunk2], axis = 1)
merged_chunks.to_csv("Data/human/salmon.merged.gene_counts.tsv", sep = "\t")

In [19]:
chunk1 = pd.read_csv("Data/mouse/salmon.merged.gene_counts1.tsv", index_col = ['gene_id', 'gene_name'], sep = "\t")
chunk2 = pd.read_csv("Data/mouse/salmon.merged.gene_counts2.tsv", index_col = ['gene_id', 'gene_name'], sep = "\t")

In [20]:
merged_chunks = pd.concat([chunk1, chunk2], axis = 1)
merged_chunks.to_csv("Data/mouse/salmon.merged.gene_counts.tsv", sep = "\t")

In [21]:
class Database:
    def __init__(self, tsv_path: str, species: str, metadata_df: pd.DataFrame, output_dir: str = "SIRT6_db"):
        """
        Initialize converter for species-combined TSV.
        
        Args:
            tsv_path: Path to combined TSV expression file for a species
            species: Species name (human, mouse, rat)
            metadata_df: DataFrame with sample metadata
            output_dir: Root directory for parquet database
        """
        self.tsv_path = Path(tsv_path)
        self.species = species.lower().replace(' ', '_')
        self.metadata_df = metadata_df.copy()
        self.output_dir = Path(output_dir)
        
        self._create_directories()
        self._process_metadata()
        self._write_statistics()

        self.gene_metadata = []
        self.experiment_stats = {}
        self.sample_metadata = {}
        
    def _create_directories(self):
        """Create the parquet database directory structure."""
        directories = [
            self.output_dir / "expression" / self.species,
            self.output_dir / "metadata" / self.species,
            self.output_dir / "genes",
            self.output_dir / "indices",
            self.output_dir / "utils"]
        
        for directory in directories:
            directory.mkdir(parents=True, exist_ok=True)
            
    def _process_metadata(self):
        self.metadata_df.rename(columns={'relation': 'sample_id',
                                    'series_id': 'experiment_id'}, 
                           inplace=True)

        samples_df = self.metadata_df[['sample_id',  
                                       'organism', 
                                       'tissue', 
                                       'genotype',
                                       'treatment',
                                       'cell_type',
                                       'sex',
                                       'strain',
                                       'age', 
                                       'condition']].copy()

        samples_df.to_parquet(self.output_dir / "metadata" / self.species / f"samples.parquet", index=False)

        experiments_df = self.metadata_df[['experiment_id', 
                                           'accession',
                                           'organism', 
                                           'extract_protocol',
                                           'contact_institute', 
                                           'instrument_model', 
                                           'library_strategy', 
                                           'title', 
                                           'summary', 
                                           'growth_protocol', 
                                           'passages']].drop_duplicates().copy()
        
        experiments_df.to_parquet(self.output_dir / "metadata" / self.species / f"experiments.parquet", index=False)
    
        self.link_df = self.metadata_df[['sample_id', 'experiment_id']].copy()
        self.link_df.to_parquet(self.output_dir / "metadata" / self.species / f"samples_to_experiment.parquet", index=False)
        
    def process_expression_data(self):
        expr_df = pd.read_csv(self.tsv_path, sep = '\t', index_col = ['gene_id', 'gene_name'])

        metadata_samples = set(self.link_df['sample_id'])
        expression_samples = set(expr_df.columns)
    
        if not metadata_samples.issubset(expression_samples):
            missing_samples = metadata_samples - expression_samples
            raise ValueError(f"Error: The following samples are in metadata but not in the expression file: {missing_samples}")
        
        valid_samples = list(metadata_samples.intersection(expression_samples))
        expr_df = expr_df[valid_samples]
        

        """
        species_dir = self.output_dir / "expression" / self.species
        experiment_expr_df.to_parquet(species_dir, 
                                      partition_cols = ['series_id', 'relation'], 
                                      index=False,
                                      engine='pyarrow')
        """
        """
        expr_df_reset = expr_df.reset_index()

        expr_df_long = pd.melt(
            expr_df_reset,
            id_vars = ['gene_id', 'gene_name'],  
            var_name = 'relation',              
            value_name = 'count'                
        )
        sample_to_gse = dict(zip(self.link_df['relation'], self.link_df['series_id']))
        expr_df_long['series_id'] = expr_df_long['relation'].map(sample_to_gse)
        
        species_dir = self.output_dir / "expression" / self.species
        expr_df_long = expr_df_long.loc[expr_df_long['count'] > 0]
        
        expr_df_long.drop(columns = ['gene_name']).to_parquet(path = species_dir,
                           partition_cols = ['series_id'],
                           engine = 'pyarrow', 
                           compression = 'zstd', 
                           compression_level = 9,
                           index = False)
        """

        self._write_gene_mappings(expr_df)
        
        grouped = self.link_df.groupby('experiment_id')
        for experiment_id, group_df in grouped:
            sample_ids = group_df['sample_id'].tolist()
            experiment_expr_df = expr_df[sample_ids]
            species_dir = self.output_dir / "expression" / self.species
            output_file_path = species_dir / f"{experiment_id}.parquet"
            experiment_expr_df.index = experiment_expr_df.index.droplevel(1)
            experiment_expr_df.to_parquet(output_file_path, compression = 'zstd', compression_level = 9)
            print(f"  -> Created {output_file_path} with {len(sample_ids)} samples.")

    def _write_gene_mappings(self, df):
        gene_mappings = pd.DataFrame.from_records(df.index.to_numpy(), 
                                                  columns = ['gene_id', 'gene_name'])
        
        species_dir = self.output_dir / "genes"
        output_file_path = species_dir / f"{self.species}_genes.parquet"
        gene_mappings.to_parquet(output_file_path, compression = 'zstd', compression_level = 9)

    def _write_statistics(self):
        experiment_stats = self.metadata_df.groupby(by = ['organism', 'genotype'], 
                                            as_index=False).agg(sample_ids = ('sample_id', list))
        experiment_stats['counts'] = [len(exp_list) for exp_list in experiment_stats.sample_ids]

        file_path = self.output_dir / "indices" / "experiment_stats.parquet"
        try:
            existing_stats = pd.read_parquet(file_path, engine = 'pyarrow')
            combined_stats = pd.concat([existing_stats, experiment_stats], ignore_index=True)

        except FileNotFoundError:
            combined_stats = experiment_stats
        
        except Exception as e:
            print(f"Error reading {file_path}. Error: {e}")
            print("Overwriting with the new DataFrame.")
            combined_stats = experiment_stats

        combined_stats.to_parquet(file_path, index = False, engine = 'pyarrow')

In [22]:
def clear_and_recreate_directory(directory_path):
    if os.path.exists(directory_path):
        shutil.rmtree(directory_path)
    os.makedirs(directory_path)

In [23]:
species = ['human', 
           'mouse', 
           'macaca', 
           'rat', 
           'drosophila', 
           'sus_scrofa']

full_organism_names = ['Homo sapiens', 
                       'Mus musculus', 
                       'Macaca fascicularis', 
                       'Rattus norvegicus', 
                       'Drosophila melanogaster', 
                       'Sus scrofa']

In [24]:
metadata = pd.read_csv("../SIRT6_datasets_metadata.csv", index_col = 0)

In [25]:
species_dict = dict.fromkeys(full_organism_names)
for name, full_name in zip(species, full_organism_names):
    species_dict[full_name] = f'Data/{name}/salmon.merged.gene_counts.tsv'

In [26]:
species_dict

{'Homo sapiens': 'Data/human/salmon.merged.gene_counts.tsv',
 'Mus musculus': 'Data/mouse/salmon.merged.gene_counts.tsv',
 'Macaca fascicularis': 'Data/macaca/salmon.merged.gene_counts.tsv',
 'Rattus norvegicus': 'Data/rat/salmon.merged.gene_counts.tsv',
 'Drosophila melanogaster': 'Data/drosophila/salmon.merged.gene_counts.tsv',
 'Sus scrofa': 'Data/sus_scrofa/salmon.merged.gene_counts.tsv'}

In [ ]:
clear_and_recreate_directory("./SIRT6_db/")
for species_name, tsv_path in species_dict.items():
    print(f"Processing {species_name}")
    metadata_species = metadata.query(f"organism == '{species_name}'")
    SIRT6_db = Database(tsv_path = tsv_path, species = species_name, metadata_df = metadata_species)
    SIRT6_db.process_expression_data()
    version_file = Path("SIRT6_db") / "version.txt"
    with open(version_file, "w") as f:
        f.write(version)

Processing Homo sapiens
  -> Created SIRT6_db/expression/homo_sapiens/GSE102813.parquet with 18 samples.
  -> Created SIRT6_db/expression/homo_sapiens/GSE212057.parquet with 24 samples.
  -> Created SIRT6_db/expression/homo_sapiens/GSE213425.parquet with 4 samples.
  -> Created SIRT6_db/expression/homo_sapiens/GSE235082.parquet with 12 samples.
  -> Created SIRT6_db/expression/homo_sapiens/GSE64642.parquet with 8 samples.
  -> Created SIRT6_db/expression/homo_sapiens/HRA003336.parquet with 6 samples.
Processing Mus musculus
  -> Created SIRT6_db/expression/mus_musculus/GSE109280.parquet with 6 samples.
  -> Created SIRT6_db/expression/mus_musculus/GSE115953.parquet with 15 samples.
  -> Created SIRT6_db/expression/mus_musculus/GSE129370.parquet with 12 samples.
  -> Created SIRT6_db/expression/mus_musculus/GSE130690,GSE130692.parquet with 6 samples.
  -> Created SIRT6_db/expression/mus_musculus/GSE157838.parquet with 28 samples.
  -> Created SIRT6_db/expression/mus_musculus/GSE166840.p